In [1]:
import time
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from pyecharts.charts import Bar3D
from pyecharts.commons.utils import JsCode
import pyecharts.options as opts
import os
os.chdir(r'D:\赵祎琳大信球\1')
sheet_names=['2015','2016','2017','2018','会员等级']
sheet_dict=pd.read_excel(r'D:\赵祎琳大信球\1\diagram\sales.xlsx',sheet_name=sheet_names)
sheet_dict

{'2015':               会员ID         订单号       提交日期    订单金额
 0      15278002468  3000304681 2015-01-01   499.0
 1      39236378972  3000305791 2015-01-01  2588.0
 2      38722039578  3000641787 2015-01-01   498.0
 3      11049640063  3000798913 2015-01-01  1572.0
 4      35038752292  3000821546 2015-01-01    10.1
 ...            ...         ...        ...     ...
 30769  39368100847  4281994827 2015-12-31   828.0
 30770       409757  4282010457 2015-12-31   199.0
 30771  38380526114  4282017675 2015-12-31   208.0
 30772     28074988  4282019440 2015-12-31    89.0
 30773  39460363230  4282025309 2015-12-31   719.0
 
 [30774 rows x 4 columns],
 '2016':               会员ID         订单号       提交日期     订单金额
 0      39288120141  4282025766 2016-01-01    76.00
 1      39293812118  4282037929 2016-01-01  7599.00
 2      27596340905  4282038740 2016-01-01   802.00
 3      15111475509  4282043819 2016-01-01    65.00
 4      38896594001  4282051044 2016-01-01    95.00
 ...            ...         ...

In [7]:
for i in sheet_names:
    print(i)
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())
for i in sheet_names[:-1]:
    sheet_dict[i]=sheet_dict[i].dropna()
    sheet_dict[i]=sheet_dict[i][sheet_dict[i]['订单金额']>1]
    sheet_dict[i]['max_year_date']=sheet_dict[i]['提交日期'].max()
for i in sheet_names:
    print(i)
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())

2015
<class 'pandas.core.frame.DataFrame'>
Int64Index: 30574 entries, 0 to 30773
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   会员ID    30574 non-null  int64         
 1   订单号     30574 non-null  int64         
 2   提交日期    30574 non-null  datetime64[ns]
 3   订单金额    30574 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 1.2 MB
None
               会员ID           订单号           订单金额
count  3.057400e+04  3.057400e+04   30574.000000
mean   2.921327e+10  4.020442e+09     967.270965
std    1.384598e+10  2.630518e+08    2073.397861
min    2.670000e+02  3.000305e+09       1.500000
25%    1.961657e+10  3.885746e+09      59.700000
50%    3.754532e+10  4.117491e+09     142.000000
75%    3.923630e+10  4.234853e+09     899.000000
max    3.954613e+10  4.282025e+09  111750.000000
2016
<class 'pandas.core.frame.DataFrame'>
Int64Index: 41001 entries, 0 to 41277
Data columns (total 4 colum

In [29]:
df_marge=pd.concat(list(sheet_dict.values())[:-1],ignore_index=True)
df_marge['year']=df_marge['提交日期'].dt.year
df_marge['date_interval']=df_marge['max_year_date']-df_marge['提交日期']
df_marge['date_interval']=df_marge['date_interval'].dt.days
rfm_df=df_marge.groupby(['会员ID','year'],as_index=False).agg({
    '订单号':'count',
    '订单金额':'sum',
    'date_interval':'min'
})
rfm_df.columns=['会员ID','year','f','m','r']
rfm_df.describe().T

,count,mean,std,min,25%,50%,75%,max
会员ID,148591.0,2.811669e+10,1.477660e+10,81.0,1.728262e+10,3.689151e+10,3.923337e+10,3.954614e+10
year,148591.0,2.016773e+03,1.129317e+00,2015.0,2.016000e+03,2.017000e+03,2.018000e+03,2.018000e+03
f,148591.0,1.365002e+00,2.626953e+00,1.0,1.000000e+00,1.000000e+00,1.000000e+00,1.300000e+02
m,148591.0,1.323741e+03,3.753907e+03,1.5,6.900000e+01,1.890000e+02,1.199000e+03,2.062518e+05
r,148591.0,1.655240e+02,1.019885e+02,0.0,7.900000e+01,1.560000e+02,2.550000e+02,3.650000e+02


In [38]:
f_bins=[0,2,5,130]
r_bins=[-1,79,255,365]
m_bins=[0,69,1199,206252]
rfm_df['f_label']=pd.cut(rfm_df['f'],bins=f_bins,labels=[i for i in range(1,len(f_bins))])
rfm_df['m_label']=pd.cut(rfm_df['m'],bins=m_bins,labels=[i for i in range(1,len(m_bins))])
rfm_df['r_label']=pd.cut(rfm_df['r'],bins=r_bins,labels=[i for i in range(len(r_bins)-1,0,-1)])

,会员ID,year,f,m,r,f_label,m_label,r_label
0,81,2016,2,159.1,2,1,2,3
1,267,2015,2,105.0,197,1,2,2
2,278,2016,2,548.5,2,1,2,3
3,278,2017,3,7137.0,67,2,3,3
4,278,2018,1,49.9,36,1,1,3
...,...,...,...,...,...,...,...,...
148586,39545536113,2016,1,2399.0,254,1,3,2
148587,39545538296,2017,1,375.0,279,1,2,1
148588,39546132904,2015,1,149.7,316,1,2,1
148589,39546134364,2015,1,49.0,242,1,1,2


In [43]:
rfm_df['f_label']=rfm_df['f_label'].astype(str)
rfm_df['m_label']=rfm_df['m_label'].astype(str)
rfm_df['r_label']=rfm_df['r_label'].astype(str)
rfm_df['rfm_group']=rfm_df['r_label']+rfm_df['f_label']+rfm_df['m_label']
rfm_df['rfm_group']
rfm_df.to_excel(r'diagram\sale_rfm_group1.xlsx',index=False)

In [51]:
display_data=rfm_df.groupby(['rfm_group','year'],as_index=False).agg({'会员ID':'count'})
display_data.columns=['rfm_group','year','number']
range_color = ['#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8', '#ffffbf',
               '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026']

range_max = int(display_data['number'].max())
c = (
    Bar3D()
    .add(
        "",
        [d.tolist() for d in display_data.values],#数据
        xaxis3d_opts=opts.Axis3DOpts(type_="category", name='分组名称'),
        yaxis3d_opts=opts.Axis3DOpts(type_="category", name='年份'),
        zaxis3d_opts=opts.Axis3DOpts(type_="value", name='会员数量'),
    )
    .set_global_opts( 
        visualmap_opts=opts.VisualMapOpts(max_=range_max, range_color=range_color), 
        title_opts=opts.TitleOpts(title="RFM分组结果"),
)
c.render() 		     

